In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *
import pyspark.sql.functions as F
catalog_name = 'ecommerce'

In [0]:
df_bronze = spark.table(f"{catalog_name}.bronze.brz_brands")
df_bronze.show(10)

In [0]:
df_silver = df_bronze.withColumn("brand_name", F.trim(F.col("brand_name")))
df_silver.show(10)

In [0]:
df_silver = df_silver.withColumn("brand_code", F.regexp_replace(F.col("brand_code"), r"[^a-zA-Z0-9]", ""))
df_silver.show(10)

In [0]:
anomalies = {
    "GROCERY": "GRCY",
    "BOOKS": "BKS",
    "TOYS": "TOY"
}

df_silver = df_silver.replace(anomalies, subset="category_code")
df_silver.show(10)

In [0]:
df_silver.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.silver.slv_brands")

In [0]:
#Category cleansing
df_bronze = spark.table(f"{catalog_name}.bronze.brz_category")
df_silver = df_bronze.dropDuplicates(['category_code'])
df_silver = df_silver.withColumn("category_code", F.upper(F.col("category_code")))

df_silver.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.silver.slv_category")

In [0]:
#Product cleansing
df_bronze = spark.table(f"{catalog_name}.bronze.brz_products")
df_silver = df_bronze.withColumn("weight_grams", F.regexp_replace(F.col("weight_grams"), "g", "").cast(IntegerType()))
df_silver = df_silver.withColumn("length_cm", F.regexp_replace(F.col("length_cm"), ",", ".").cast(FloatType()))
df_silver = df_silver.withColumn("category_code", F.upper(F.col("category_code"))) \
                    .withColumn("brand_code", F.upper(F.col("brand_code")))
df_silver = df_silver.withColumn("material", F.when(F.col("material") == "Coton", "Cotton") \
                                        .when(F.col("material") == "Alumium", "Aluminium") \
                                        .when(F.col("material") == "Ruber", "Rubber") \
                                        .otherwise(F.col("material")))
df_silver = df_silver.withColumn("rating_count", F.when(F.col("rating_count").isNotNull(), F.abs(F.col("rating_count"))) \
                                        .otherwise(F.lit(0)))

df_silver.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.silver.slv_products")

In [0]:
#Customers cleansing
df_bronze = spark.table(f"{catalog_name}.bronze.brz_customers")
df_silver = df_bronze.dropna(subset=["customer_id"])
df_silver = df_silver.fillna("Not Available", subset=["phone"])


df_silver.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.silver.slv_customers")

In [0]:
#Date cleansing
df_bronze = spark.table(f"{catalog_name}.bronze.brz_date")
df_silver = df_bronze.withColumn("date", to_date(df_bronze["date"], "dd-MM-yyyy"))
df_silver = df_silver.dropDuplicates(['date'])
df_silver = df_silver.withColumn("day_name", F.initcap(F.col("day_name")))
df_silver = df_silver.withColumn("week_of_year", F.abs(F.col("week_of_year")))
df_silver = df_silver.withColumn("quarter", F.concat_ws("", F.concat(F.lit("Q"), F.col("quarter"), F.lit("-"), F.col("year"))))
df_silver = df_silver.withColumn("week_of_year", F.concat_ws("-", F.concat(F.lit("Week"), F.col("week_of_year"), F.lit("-"), F.col("year"))))
df_silver = df_silver.withColumnRenamed("week_of_year", "week")


df_silver.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.silver.slv_calendar")